# 06_explainable_ai

SHAP global and local explanations.

---

*DiaPredict AI — Explainable Diabetes Risk Prediction System.* *Educational / research prototype, not a medical diagnosis.*

## Objectives
- Load the trained XGBoost surrogate
- Compute **global** feature importance (mean |SHAP|)
- Compute a **local** explanation for a single patient
- Understand what the SHAP values mean

In [ ]:
import sys
from pathlib import Path

# Make the project root importable no matter where the notebook runs.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Root:", PROJECT_ROOT)


In [ ]:
import numpy as np
from backend.app.preprocessing import load_raw_dataset, make_train_test
from backend.app.model_loader import get_registry

raw = load_raw_dataset()
X_train, y_train, X_test, y_test, scaler, medians = make_train_test(raw)
print("Background matrix:", X_train.shape)

### 1. Global feature importance (mean |SHAP|)

In [ ]:
from backend.app.explainability import get_explainer

explainer = get_explainer()
global_imp = explainer.global_feature_importance(X_train[:200])
import pandas as pd
gdf = pd.DataFrame(global_imp)
print(gdf)
gdf.plot(kind="barh", x="label", y="mean_abs_shap", legend=False, color="#60a5fa", figsize=(9, 5))
plt.title("Global feature importance (mean |SHAP|)"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### 2. Local explanation for one patient

In [ ]:
from backend.app.schemas import PatientInput

patient = PatientInput(
    Pregnancies=6, Glucose=148, BloodPressure=72, SkinThickness=35,
    Insulin=155, BMI=33.6, DiabetesPedigreeFunction=0.627, Age=50)

# Scale the patient's features with the persisted scaler/medians.
from backend.app.services import prepare_feature_vector
X_patient = prepare_feature_vector(patient)
base_prob = 0.71  # illustrative; the backend computes this live

local = explainer.local_explanation(X_patient, base_prob)
print("Summary:", local["summary_english"])
print()
print(f"{'Feature':28s}{'SHAP':>10s}{'Contribution':>14s}")
for b in local["top_features"]:
    print(f"{b['label']:28s}{b['value']:>10.3f}{b['contribution']:>14.3f}")

### 3. Waterfall-style plot of contributions

In [ ]:
bars = local["top_features"]
features = [b["label"] for b in bars][::-1]
contrib = [b["contribution"] for b in bars][::-1]
colors = ["#f87171" if c > 0 else "#60a5fa" for c in contrib]
plt.figure(figsize=(9, 5))
plt.barh(features, contrib, color=colors)
plt.axvline(0, color="black", lw=1)
plt.xlabel("Contribution to predicted probability")
plt.title("What influenced this prediction?")
plt.tight_layout(); plt.show()

### What does a SHAP value mean?

A SHAP value measures the contribution of a feature to the difference between the model's prediction for *this* patient and the model's average prediction. Positive values push the prediction higher (towards diabetes); negative values push it lower. Shown in **probability space**, a value of `+0.2` means that feature increased the predicted probability by about 0.2.

> **Why explain?** Explainability is what makes a medical-risk model auditable and trustworthy. It tells a user which factors drove the estimate, so a clinician can interpret and sanity-check the output — which is a key requirement for any AI system used in a health context.